# Daniel: re-train C′ and D (more seeds) and train D2

**Run All once.** Each step is its own cell, so you can follow the progress.
If it stops (hang, power loss), just **Run All** again: finished steps and models are skipped
and an interrupted model continues from its last epoch.

| Step | What | Time |
|---|---|---|
| 1 | D2 calibration: β per skin tone from lesion visibility (train images only) | ~3–5 min (skipped if already done) |
| 2 | Prepare the D2 dataset | ~2 min (skipped if already done) |
| 3 | Train C′ and D (seeds 43, 44) and D2 (seeds 42, 43, 44) | ~45 min per model and seed |
| 4 | Compare C′, D, D2 on validation and test (mean ± SD) | ~20 min |

**Tips so the laptop doesn't hang:**
- Run only **one** training at a time (don't run a script or another notebook at the same time).
- Close heavy apps (browser tabs, games). Keep the laptop plugged in.

Results: `ablation_training/runs/ablation_split/d2_seeds/comparison_summary.md`

In [ ]:
import os, sys, importlib
from pathlib import Path
os.environ.setdefault('MPLBACKEND', 'Agg')
HERE = Path.cwd().resolve()
if not (HERE / 'retrain_c2_d_d2.py').is_file():
    HERE = HERE / 'daniel'
assert (HERE / 'retrain_c2_d_d2.py').is_file(), 'Open this notebook from ablation_training/daniel.'
sys.path.insert(0, str(HERE))
import retrain_c2_d_d2 as r
importlib.reload(r)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
assert torch.cuda.is_available(), 'No GPU: select the TOOL-26 (.venv) kernel.'
print('Results folder:', r.OUT)

## Step 1: D2 calibration (β per skin tone)

In [ ]:
d2 = r.calibrate_d2()
print('\nD2 betas:', d2['betas'])
for bracket, reason in d2['reasons'].items():
    print(f'  {bracket}: {reason}')

## Step 2: Prepare datasets (C′, D, D2)

In [ ]:
ns = r.notebook_namespace(d2['betas'])
print('Ready:', ns['TRAIN_MODELS'], 'seeds', ns['SEEDS'])

## Step 3: Train C′, D (seeds 43, 44) and D2 (seeds 42, 43, 44)

Finished runs are skipped. The progress bar shows the epoch; each model stops by itself
(early stopping, patience 50).

In [ ]:
exec(compile(ns['_training_source'], 'training', 'exec'), ns)

## Step 4: Compare C′, D, D2 (validation = decide, test = final report)

In [ ]:
r.evaluate_all(ns)
print('\nSaved:', r.OUT / 'comparison_summary.md')

## How to read the tables
- **Validation:** use this to decide whether D2 is better than C′ and D.
- **Test:** the official numbers for the manuscript.
- **Darkest skin:** the main claim of the thesis.
- A difference smaller than the ± SD is not a real difference.